In [ ]:
import json
import requests

from google.colab import userdata
from io import BytesIO
from os import makedirs
from PIL import Image as PImage
from time import sleep

# make an empty directory to download images into
makedirs("images/500", exist_ok=True)

# when running on colab, this comes from the "secrets" panel on the left
SI_KEY = userdata.get("SI_KEY")

SI_URL = "https://api.si.edu/openaccess/api/v1.0/search"

In [ ]:
# number of objects to get per query
OBJS_PER_QUERY = 1000

# term to search for
QUERY_TERM = "astrolabe"
# QUERY_TERM = "orchid"
# QUERY_TERM = "political poster"

# query and the other params
params = {
  "q": f"{QUERY_TERM} AND online_media",
  "api_key": SI_KEY,
  "start": 0,
  "sort": "id",
  "rows": 0,
}

# get a response and decode json into variable
response = requests.get(SI_URL, params=params)
data = response.json()

# total number of objects
n_rows = data["response"]["rowCount"]

# empty list to store objects
objects = []

# loop to get objects 1000 rows at a time
for cnt in range(n_rows // OBJS_PER_QUERY + 1):
  # update params for the number of rows and the starting point
  params["start"] = cnt * OBJS_PER_QUERY
  params["rows"] = OBJS_PER_QUERY

  # get a response and decode json into variable
  response = requests.get(SI_URL, params=params)
  data = response.json()

  # iterate over returned objects/rows
  for row in data["response"]["rows"]:
    # counter for current object index
    ocnt = len(objects)

    # print progress and save json every 25 objects
    if ocnt % 25 == 0:
      print(ocnt, "/", n_rows)
      with open("data.json", "w") as ofp:
        json.dump(objects, ofp)

    # try to extract image info from nested object
    try:
      # if some of the properties are missing, this will throw an error
      row_media = row["content"]["descriptiveNonRepeating"]["online_media"]["media"][0]
    except:
      # and we assume the image info is missing
      row_media = {}

    # if there's a thumbnail
    if "thumbnail" in row_media:
      # try to download the image
      try:
        # download image, resize it and save it
        response = requests.get(row_media["thumbnail"])
        response.raise_for_status()
        img = PImage.open(BytesIO(response.content)).convert("RGB")
        img.thumbnail((500, 500))
        img.save(f"images/500/{row['id']}.jpg")

        # try to access objects's data_source
        try:
          source = row["content"]["descriptiveNonRepeating"]["data_source"]
        except:
          # if some properties don't exist, set it to an empty string
          source = ""

        # try to access objects's description
        try:
          description = row["content"]["freetext"]["notes"][0]["content"]
        except:
          # if some properties don't exist, set it to an empty string
          description = ""

        # new object with just the info we want
        my_obj = {
          "id": row["id"],
          "url": row["url"],
          "source": source,
          "description": description,
          "first_image": row_media["thumbnail"]
        }
        # add to list that gets saved as json
        objects.append(my_obj)

      # if we can't download the image, skip this row
      except requests.exceptions.HTTPError as errh:
        pass

    # slow down to prevent getting kicked out of SI's API
    sleep(0.05)
  sleep(0.1)

print(len(objects))

In [ ]:
# save final version of the json
with open("data.json", "w") as ofp:
  json.dump(objects, ofp)

In [ ]:
# compresses the directory with all of the images
!tar -czf images.tgz images/